# Qwen-VL Training Visualization

This notebook visualizes training metrics from `train_metrics.jsonl` and `val_metrics.jsonl`.

In [ ]:
import json
import os
import matplotlib.pyplot as plt

def load_jsonl(filepath):
    """Load metrics from jsonl file."""
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return []
    metrics = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                metrics.append(json.loads(line))
    return metrics

In [ ]:
# Configuration
EXP_NAME = "qwen3vl_finetune"  # Change this to your experiment name
SAVE_DIR = "./outputs"

EXP_DIR = os.path.join(SAVE_DIR, EXP_NAME)
LOG_DIR = os.path.join(EXP_DIR, "log")
PLT_DIR = os.path.join(EXP_DIR, "plt_fig")

os.makedirs(PLT_DIR, exist_ok=True)

train_metrics_path = os.path.join(LOG_DIR, "train_metrics.jsonl")
val_metrics_path = os.path.join(LOG_DIR, "val_metrics.jsonl")

In [ ]:
# Load metrics
train_metrics = load_jsonl(train_metrics_path)
val_metrics = load_jsonl(val_metrics_path)

print(f"Loaded {len(train_metrics)} train metric records")
print(f"Loaded {len(val_metrics)} val metric records")

In [ ]:
# Extract training loss
if train_metrics:
    train_steps = [m['step'] for m in train_metrics if 'step' in m and 'loss' in m]
    train_losses = [m['loss'] for m in train_metrics if 'step' in m and 'loss' in m]
    print(f"Train steps: {len(train_steps)}")
    print(f"First train loss: {train_losses[0] if train_losses else 'N/A'}")
    print(f"Last train loss: {train_losses[-1] if train_losses else 'N/A'}")

In [ ]:
# Plot training loss
if train_steps:
    plt.figure(figsize=(10, 6))
    plt.plot(train_steps, train_losses, 'b-', alpha=0.7, label='Train Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title(f'Training Loss - {EXP_NAME}')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    loss_curve_path = os.path.join(PLT_DIR, "loss_curve.png")
    plt.savefig(loss_curve_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Loss curve saved to: {loss_curve_path}")

In [ ]:
# Extract validation metrics
if val_metrics:
    val_steps = [m['step'] for m in val_metrics if 'step' in m]
    val_losses = [m.get('val_loss', 0) for m in val_metrics if 'step' in m]
    clipscores = [m.get('clipscore', 0) for m in val_metrics if 'step' in m]
    print(f"Val steps: {len(val_steps)}")
    if val_losses:
        print(f"First val loss: {val_losses[0]}")
        print(f"Last val loss: {val_losses[-1]}")

In [ ]:
# Plot validation metrics
if val_steps:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Val loss
    axes[0].plot(val_steps, val_losses, 'r-', marker='o', label='Val Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Validation Loss')
    axes[0].set_title(f'Validation Loss - {EXP_NAME}')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # CLIPScore (if available)
    if any(clipscores):
        axes[1].plot(val_steps, clipscores, 'g-', marker='s', label='CLIPScore')
        axes[1].set_xlabel('Step')
        axes[1].set_ylabel('CLIPScore')
        axes[1].set_title(f'CLIPScore - {EXP_NAME}')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()
    
    plt.tight_layout()
    
    val_curve_path = os.path.join(PLT_DIR, "val_curve.png")
    plt.savefig(val_curve_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Val curve saved to: {val_curve_path}")

In [ ]:
print("Visualization complete!")
print(f"Output directory: {PLT_DIR}")